# Storage-Agnostic KG Building: one pipeline, any backend

The same `Baseline` pipeline builds the knowledge graph **wherever you want
it stored** — the pipeline code never changes. Polygraph separates the
*what to build* (the pipeline) from the *where to store it* (the backend):

- **Write side** — `build_kg_into(writer, chunks, entities, triples)` writes
  through any `GraphWriter`: `NetworkXGraphWriter` (in-memory python object),
  `SQLiteGraphWriter` (file-backed), `Neo4jGraphBuilder` (streamed to Neo4j).
  One build routine, swap the writer.
- **Read side** — every backend is exposed through the same `GraphStore`
  interface (`number_of_nodes`, `nodes`, `get_node`, `successors`, `search`,
  ...). Get one from `kg["graph_store"]` or `create_graph_store(backend)`.

This notebook builds the **exact same pipeline** twice — once **locally**
(in-memory `networkx`) and once in **Neo4j** — and interrogates both graphs
with identical code. Only `graph_store_backend` changes.

**Prerequisites:** the Neo4j cells require a running instance and
`NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` in the environment (or `.env`).
The local cells run with no extra services.

In [ ]:
# One pipeline definition — only the storage backend differs between runs.
from polygraph._shared.stage_config import ExtractionConfig, PreprocessConfig
from polygraph.pipelines import Baseline

# Offline-safe knobs (sentence chunking + minhash dedup + regex extraction
# avoid HuggingFace/spaCy model downloads so the tutorial runs anywhere).
SHARED = {
    "preprocess": PreprocessConfig(
        quality_min_chars=50,
        quality_min_words=10,
        chunk_method="sentence",
        doc_dedup_method="minhash",
        chunk_dedup_method="minhash",
    ),
    "extraction": ExtractionConfig(
        mode="composed", entity_method="regex", relation_method="ontology_rules"
    ),
}

In [ ]:
# 1) Build LOCALLY: the graph is an in-memory networkx.DiGraph.
local_pipe = Baseline(graph_store_backend="networkx", **SHARED)
kg_local = local_pipe.execute(
    input_paths=["data/wikipedia/connected.jsonl"],
    output_dir="output/neo4j_upload_tutorial/local",
)

store_local = kg_local["graph_store"]  # NetworkXGraphStore
print(f"backend: {type(store_local).__name__}")
print(f"local   → {store_local.number_of_nodes()} nodes, {store_local.number_of_edges()} edges")
print("sample nodes:", [nid for nid, _ in list(store_local.nodes(data=True))[:5]])

In [ ]:
# 2) The SAME pipeline, same config — only the storage backend differs.
#    With graph_store_backend="neo4j" the graph is streamed straight into
#    Neo4j during the build: no in-memory graph is ever materialised
#    (kg["graph"] is None), and file export is skipped automatically.
neo4j_pipe = Baseline(
    graph_store_backend="neo4j",
    graph_store_options={"clear": True},  # wipe Neo4j first; False = append
    **SHARED,
)
kg_neo4j = neo4j_pipe.execute(
    input_paths=["data/wikipedia/connected.jsonl"],
    output_dir="output/neo4j_upload_tutorial/neo4j",
)

print("streamed to Neo4j:", kg_neo4j.get("neo4j_stats"))
store_neo4j = kg_neo4j["graph_store"]  # live Neo4jGraphStore
print(f"neo4j   → {store_neo4j.number_of_nodes()} nodes, {store_neo4j.number_of_edges()} edges")

In [ ]:
# 3) The read side is storage-agnostic too: both stores answer the SAME
#    GraphStore API, so downstream code never cares where the graph lives.
for label, store in [("local", store_local), ("neo4j", store_neo4j)]:
    print(
        f"{label:6} {type(store).__name__:20} "
        f"{store.number_of_nodes():>4} nodes, {store.number_of_edges():>4} edges"
    )

# Identical queries against both backends: has_node / get_node / out_degree...
first_node = next(iter(store_local.nodes()))
print("\nsame GraphStore API — first node:", first_node)
print("has_node :", store_local.has_node(first_node), "|", store_neo4j.has_node(first_node))
print("out_degree:", store_local.out_degree(first_node), "|", store_neo4j.out_degree(first_node))

In [ ]:
# Optional: verify Neo4j directly with Cypher — independent of the GraphStore.
import os

from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ.get("NEO4J_URI", "bolt://localhost:7687"),
    auth=(
        os.environ.get("NEO4J_USER", "neo4j"),
        os.environ.get("NEO4J_PASSWORD", ""),
    ),
)

with driver.session() as session:
    nodes = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    rels = session.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    print(f"Neo4j holds {nodes} nodes and {rels} relationships")

driver.close()

## How the abstraction works

- **Write side (`GraphWriter` + `build_kg_into`)** — one routine derives the
  document/chunk nodes, `PART_OF`/`NEXT` edges, entity nodes, and semantic
  edges, then writes them through any `GraphWriter`. Add a new storage
  backend by implementing 5 methods (`merge_document`, `merge_chunk`,
  `merge_entity`, `merge_edge`, `merge_structural_edge`).
- **Read side (`GraphStore` + `create_graph_store`)** — one interface
  (`nodes`, `edges`, `get_node`, `has_edge`, `successors`, `in_degree`,
  `search`, `to_networkx`) reads a graph no matter where it lives:
  `create_graph_store("networkx" | "sqlite" | "neo4j" | "json" | "graphml")`.

## Choosing a backend

| `graph_store_backend` | Where the graph lives | Memory |
|---|---|---|
| `"networkx"` (default) | in-memory `nx.DiGraph` | all nodes + edges in RAM |
| `"sqlite"` | `knowledge_graph.db` file | queried data only |
| `"neo4j"` | Neo4j database (streamed) | none in Python |
| `"json"` / `"graphml"` | exported file, read back | lazy via file |

- With `"neo4j"`, `kg["graph"]` is `None` and file export is skipped — the
  graph only exists in Neo4j. Use `graph_store_options={"clear": True}` to
  wipe first or `False` to append onto an existing graph.
- `Baseline` preprocesses the whole corpus in RAM. For corpora too large for
  that, `StreamingPipeline(...).execute_batched(batch_size=N)` streams
  documents in fixed-size batches (at the cost of cross-batch dedup and
  entity resolution).
- To inspect the local graph as a plain object: `store_local.to_networkx()`.